> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t07_backfill_raw_to_bronze` · notebook `01_raw_to_bronze` · started 2026-10-11 13:05 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `drift_mode=evolve`, `end_date=2026-10-08`, `load_type=all`, `mode=backfill`, `start_date=2026-09-25`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 01 · Raw → Bronze
Loads landing batches (JSONL + `_manifest.json`) into the Bronze Delta tables using **explicit StructType
schemas** (no inference). Invalid records go to `so_bronze.quarantine_records`, schema drift is recorded in
`so_ops.schema_drift_events`, and every file processed writes a row to `so_ops.pipeline_execution_logs`.

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: every batch not yet loaded | `backfill`: reload the selected batches |
| `load_type` | `all` | `all`, `full` or `incremental` |
| `batch_ids` | empty | e.g. `incr_20260925T155859Z,full_2025-01-01_2025-04-01` |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `landing_path` | empty | an explicit batch folder, e.g. `/Volumes/workspace/so_raw/landing/incremental/incr_...` |
| `drift_mode` | `evolve` | `evolve` adds new columns (mergeSchema); `rescue` keeps them in `_rescued_data` |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.dropdown("load_type", "all", ["all", "full", "incremental"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("landing_path", "")
dbutils.widgets.dropdown("drift_mode", "evolve", ["evolve", "rescue"])

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.bronze import run_raw_to_bronze

cfg = build_config()
audit = run_raw_to_bronze(
    spark, cfg,
    mode=widget("mode"),
    load_type=widget("load_type"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    landing_path=widget("landing_path") or None,
    fail_on_error=True,
)

Raw-to-Bronze: 6 batch(es) selected, mode=backfill
[SUCCESS] Raw-to-Bronze | questions                          | read=2738 ins=2738 upd=0 del=2738 quar=0 | 24.0s | replaced 2738 rows from a previous load of this batch
[SUCCESS] Raw-to-Bronze | answers                            | read=3384 ins=3384 upd=0 del=3384 quar=0 | 11.6s | replaced 3384 rows from a previous load of this batch
[SUCCESS] Raw-to-Bronze | questions                          | read=4548 ins=4548 upd=0 del=4548 quar=0 | 10.6s | replaced 4548 rows from a previous load of this batch
[SUCCESS] Raw-to-Bronze | answers                            | read=5528 ins=5528 upd=0 del=5528 quar=0 | 10.8s | replaced 5528 rows from a previous load of this batch
[SUCCESS] Raw-to-Bronze | questions                          | read=35 ins=35 upd=0 del=35 quar=0 | 9.5s | replaced 35 rows from a previous load of this batch
[SUCCESS] Raw-to-Bronze | answers                            | read=59 ins=59 upd=0 del=59 quar=0 | 8.0s | replaced 59

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
49aaa037-911b-4def-96c6-e651a9f3b1fc,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,full,questions,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/questions.jsonl,workspace.so_bronze.questions,2026-10-11T13:05:27.541Z,2026-10-11T13:05:51.515Z,23.974149,SUCCESS,2738,2738,0,2738,0,replaced 2738 rows from a previous load of this batch,2026-10-11T13:05:51.515Z
375ab358-b5e3-4205-81d6-4adf7d606ff2,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,full,answers,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/answers.jsonl,workspace.so_bronze.answers,2026-10-11T13:05:53.304Z,2026-10-11T13:06:04.897Z,11.593185,SUCCESS,3384,3384,0,3384,0,replaced 3384 rows from a previous load of this batch,2026-10-11T13:06:04.897Z
8aea0523-58e5-464b-9187-b9556267e828,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,full,questions,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/questions.jsonl,workspace.so_bronze.questions,2026-10-11T13:06:15.172Z,2026-10-11T13:06:25.767Z,10.594777,SUCCESS,4548,4548,0,4548,0,replaced 4548 rows from a previous load of this batch,2026-10-11T13:06:25.767Z
60f80f56-5532-46ef-9422-1dd5d4c29c81,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,full,answers,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/answers.jsonl,workspace.so_bronze.answers,2026-10-11T13:06:27.261Z,2026-10-11T13:06:38.082Z,10.820422,SUCCESS,5528,5528,0,5528,0,replaced 5528 rows from a previous load of this batch,2026-10-11T13:06:38.082Z
8cab1f89-0095-4b5f-a482-164fd9d5188b,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,incremental,questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/questions.jsonl,workspace.so_bronze.questions,2026-10-11T13:06:46.723Z,2026-10-11T13:06:56.258Z,9.53554,SUCCESS,35,35,0,35,0,replaced 35 rows from a previous load of this batch,2026-10-11T13:06:56.258Z
2de2f282-65a5-491d-b8fa-77fba2433936,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,incremental,answers,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/answers.jsonl,workspace.so_bronze.answers,2026-10-11T13:06:57.785Z,2026-10-11T13:07:05.826Z,8.040811,SUCCESS,59,59,0,59,0,replaced 59 rows from a previous load of this batch,2026-10-11T13:07:05.826Z
54d799f9-8df3-4c0f-922a-d402f26aaa4e,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,incremental,deleted_questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/deleted_questions.jsonl,workspace.so_bronze.deleted_questions,2026-10-11T13:07:07.561Z,2026-10-11T13:07:12.641Z,5.080323,SUCCESS,0,0,0,0,0,,2026-10-11T13:07:12.641Z
d5fca769-0ed9-42ae-ba47-e781543987d5,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,incremental,question_snapshots,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/question_snapshots.jsonl,workspace.so_bronze.question_snapshots,2026-10-11T13:07:14.210Z,2026-10-11T13:07:21.019Z,6.808906,SUCCESS,100,100,0,100,0,replaced 100 rows from a previous load of this batch,2026-10-11T13:07:21.019Z
8f323c15-3dc5-4c61-b3df-9c85ea7c8c03,raw_to_bronze-20261011T130525Z-1ec894,raw_to_bronze,Raw-to-Bronze,backfill,incremental,questions,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z/questions.jsonl,workspace.so_bronze.questions,2026-10-11T13:07:31.724Z,2026-10-11T13:07:39.412Z,7.688486,SUCCESS,305,305,0,30